# Set up reusable PubMed tools for Claude — verification notebook

**Companion notebook** for the rendered note at
👉 [tensoromics.com/notes/claude-tools-getting-started](https://tensoromics.com/notes/claude-tools-getting-started)

**This notebook is the verification path, not the lesson.** The lesson lives in the rendered note — building two Python files (`~/claude-tools/claude_runtime.py` and `~/claude-tools/pubmed_tools.py`) and a Skill (`~/.claude/skills/pubmed-tools/SKILL.md`), then installing them on `PYTHONPATH`. Once that's done, the cells below import the installed modules and confirm that all three demonstrations work.

If any cell errors, the rendered note has a Troubleshooting section. The most common failure is `ModuleNotFoundError` — meaning `PYTHONPATH` doesn't include `~/claude-tools/` in this kernel's environment. Fix: source `.zshrc` and restart the kernel.

> **Editing workflow.** The four code cells below are mirrored byte-for-byte in the corresponding code blocks of the MDX (sections 7 and 8). The `.py` file-content blocks in the MDX are reference material for copying to disk, not cells in this notebook — that's intentional, because the whole point of the install is to import from `~/claude-tools/` rather than redefine inline.

## Prerequisites

Before running the cells below, you should have:

1. Done the [Claude API setup](https://tensoromics.com/notes/claude-api-setup/) — venv with `anthropic` + `requests` installed, `ANTHROPIC_API_KEY` in `.env` and loaded via `python-dotenv`.
2. Saved `~/claude-tools/claude_runtime.py` and `~/claude-tools/pubmed_tools.py` per sections 4 and 5 of the rendered note.
3. Added `export PYTHONPATH="$HOME/claude-tools:$PYTHONPATH"` to `~/.zshrc` and sourced it (or opened a fresh terminal).
4. Confirmed `python -c "from pubmed_tools import TOOLS; print(len(TOOLS))"` prints `2` from your shell.

If you launched JupyterLab BEFORE adding the `PYTHONPATH` export, restart the lab from a fresh terminal so the kernel inherits the updated environment.

In [ ]:
# Load the API key from .env if you're using python-dotenv
from dotenv import load_dotenv
load_dotenv()

## Verification — does the install work?

Three import lines, one function call. This matches the verification example in section 7 of the rendered note.

In [ ]:
from pubmed_tools   import TOOLS, TOOL_FUNCS
from claude_runtime import chat_with_tools

print(chat_with_tools("How many papers on KRAS G12C inhibitors were published in 2026?", TOOLS, TOOL_FUNCS))

**Expected** (wording will vary; the number is live from PubMed): a prose answer that quotes a count in the hundreds or low thousands and adds a brief comment about field activity. If you got an answer, the API path is fully wired.

## Demonstration 1 — direct call, no Claude

When you just want the function output:

In [ ]:
from pubmed_tools import pubmed_count

n = pubmed_count("KRAS G12C inhibitor", year=2026)
print(f"PubMed papers on KRAS G12C inhibitors in 2026: {n}")

## Demonstration 2 — conversational, Claude picks `pubmed_search`

Different question shape: asking for recent papers (not a count). Claude reads the tool descriptions and picks the right one.

In [ ]:
print(chat_with_tools("Show me the 3 most recent PubMed papers on PROTAC degraders.", TOOLS, TOOL_FUNCS))

## Demonstration 3 — chained, both tools in one conversation

A question that naturally needs both `pubmed_count` and `pubmed_search`. Claude figures out the sequence.

In [ ]:
print(chat_with_tools(
    "How active is spatial transcriptomics — how many papers in 2026 so far, "
    "and show me the 3 most recent ones?",
    TOOLS, TOOL_FUNCS
))

## Done

If all three demonstrations returned sensible prose with live PubMed numbers, the install is verified. The `.py` files are on `PYTHONPATH`, the API key is loaded, and any future notebook in any project can import the same modules with the same three-line pattern.

For Claude Code (the `claude` CLI) verification — that the Skill at `~/.claude/skills/pubmed-tools/SKILL.md` works — open a fresh terminal, run `claude`, and ask a literature question in plain English. That part doesn't live in this notebook; it's section 9 of the rendered note.

Any future tool module you write goes in `~/claude-tools/` alongside `pubmed_tools.py`; the same install pattern carries it from any notebook or Claude Code session.